# Exploratory Data Analysis - Predict Future Sales

## 2. Comprensión inicial del dataset

### 2.a Carga de los archivos y primeras filas

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW = Path("..") / "data" / "raw"

sales = pd.read_csv(RAW / "sales_train.csv")
items = pd.read_csv(RAW / "items.csv")
item_categories = pd.read_csv(RAW / "item_categories.csv")
shops = pd.read_csv(RAW / "shops.csv")
test = pd.read_csv(RAW / "test.csv")

# Copia de la fecha original en texto, para validar el formato en 3.b.
fecha_texto = sales["date"].copy()

tablas = {
    "sales_train": sales,
    "items": items,
    "item_categories": item_categories,
    "shops": shops,
    "test": test,
}

In [2]:
sales.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,02.01.2013,0,59,22154,999.00,1.0
1,03.01.2013,0,25,2552,899.00,1.0
2,05.01.2013,0,25,2552,899.00,-1.0
3,06.01.2013,0,25,2554,1709.05,1.0
4,15.01.2013,0,25,2555,1099.00,1.0


**Interpretación:** cada fila es una transacción de un producto en una tienda y un día. La fecha viene como texto `dd.mm.aaaa`. Ya en las primeras filas aparece una devolución: el producto 2552 de la tienda 25 se vende el 03.01.2013 y tiene `item_cnt_day = -1` el 05.01.2013.

In [3]:
items.head()

,item_name,item_id,item_category_id
0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,0,40
1,!ABBYY FineReader 12 Professional Edition Full...,1,76
2,***В ЛУЧАХ СЛАВЫ (UNV) D,2,40
3,***ГОЛУБАЯ ВОЛНА (Univ) D,3,40
4,***КОРОБКА (СТЕКЛО) D,4,40


**Interpretación:** es el catálogo que une cada `item_id` con su categoría. Los nombres están en ruso y traen ruido de texto (símbolos como `!` y `***`, y espacios de relleno), por lo que `item_category_id` es más útil que `item_name` para agrupar productos.

In [4]:
item_categories.head()

,item_category_name,item_category_id
0,PC - Гарнитуры/Наушники,0
1,Аксессуары - PS2,1
2,Аксессуары - PS3,2
3,Аксессуары - PS4,3
4,Аксессуары - PSP,4


**Interpretación:** los nombres siguen el patrón `familia - subcategoría` (por ejemplo, `Аксессуары - PS2`, accesorios de PS2), lo que permitiría separar una categoría más general en etapas posteriores.

In [5]:
shops.head()

,shop_name,shop_id
0,"!Якутск Орджоникидзе, 56 фран",0
1,"!Якутск ТЦ ""Центральный"" фран",1
2,"Адыгея ТЦ ""Мега""",2
3,"Балашиха ТРК ""Октябрь-Киномир""",3
4,"Волжский ТЦ ""Волга Молл""",4


**Interpretación:** el nombre de cada tienda empieza con la ciudad (Якутск, Адыгея, Балашиха, Волжский), seguida de datos del centro comercial. Las tiendas 0 y 1 están en Якутск y llevan `!` al inicio, un detalle a revisar cuando se busquen tiendas repetidas en la sección 4.

In [6]:
test.head()

,ID,shop_id,item_id
0,0,5,5037
1,1,5,5320
2,2,5,5233
3,3,5,5232
4,4,5,5268


**Interpretación:** `test` solo contiene los pares tienda-producto a predecir con su `ID`. No trae fecha, precio ni cantidad, así que toda la información para el modelo debe salir del histórico.

### 2.b Tamaño de cada tabla

In [7]:
pd.DataFrame(
    [{"tabla": nombre, "filas": df.shape[0], "columnas": df.shape[1]} for nombre, df in tablas.items()]
).set_index("tabla")

,filas,columnas
tabla,,
sales_train,2935849,6
items,22170,3
item_categories,84,2
shops,60,2
test,214200,3


**Interpretación:** el histórico tiene 2,935,849 transacciones, frente a catálogos pequeños de 22,170 productos, 84 categorías y 60 tiendas. `test` tiene 214,200 filas, que equivalen exactamente a 42 tiendas × 5,100 productos (ver 3.a): se pide predecir la combinación completa de esas tiendas y productos, aunque un par no se haya vendido nunca.

### 2.c Descripción de las columnas y variable objetivo

Descripción de las columnas de las tablas del dataset:

| Columna | Tabla(s) | Tipo | Descripción |
|---|---|---|---|
| `date` | `sales_train` | Fecha (texto `dd.mm.aaaa` en el archivo) | Día en que ocurrió la transacción. |
| `date_block_num` | `sales_train` | Entero ordinal | Número de mes consecutivo: 0 = enero 2013, 33 = octubre 2015. |
| `shop_id` | `sales_train`, `shops`, `test` | Identificador categórico | Identificador único de la tienda (sucursal). |
| `item_id` | `sales_train`, `items`, `test` | Identificador categórico | Identificador único del producto. |
| `item_price` | `sales_train` | Numérica continua | Precio unitario del producto en esa transacción. |
| `item_cnt_day` | `sales_train` | Numérica discreta | Unidades vendidas en el día; los valores negativos representan devoluciones. |
| `item_name` | `items` | Texto | Nombre del producto. |
| `item_category_id` | `items`, `item_categories` | Identificador categórico | Identificador de la categoría del producto. |
| `item_category_name` | `item_categories` | Texto | Nombre de la categoría del producto. |
| `shop_name` | `shops` | Texto | Nombre de la tienda. |
| `ID` | `test` | Identificador | Identificador de cada par tienda-producto a predecir. |

**Variable objetivo:** la suma mensual de `item_cnt_day` por tienda-producto, es decir, las unidades vendidas de cada par (`shop_id`, `item_id`) en cada mes (`date_block_num`). No existe como columna en los archivos: se construye agregando `sales_train`, y debe predecirse para noviembre de 2015 en los pares de `test.csv`.

### 2.d Rango temporal de los datos

In [8]:
fechas = pd.to_datetime(fecha_texto, format="%d.%m.%Y")

pd.Series(
    {
        "fecha mínima": fechas.min().date(),
        "fecha máxima": fechas.max().date(),
        "días en el rango": (fechas.max() - fechas.min()).days + 1,
        "meses calendario distintos": fechas.dt.to_period("M").nunique(),
        "date_block_num mínimo": sales["date_block_num"].min(),
        "date_block_num máximo": sales["date_block_num"].max(),
        "date_block_num distintos": sales["date_block_num"].nunique(),
    },
    name="valor",
).to_frame()

,valor
fecha mínima,2013-01-01
fecha máxima,2015-10-31
días en el rango,1034
meses calendario distintos,34
date_block_num mínimo,0
date_block_num máximo,33
date_block_num distintos,34


**Interpretación:** los datos van del 1 de enero de 2013 al 31 de octubre de 2015, es decir, 1,034 días y 34 meses calendario. `date_block_num` toma exactamente 34 valores (0 a 33), como indica el PDF, y el mes a predecir (noviembre de 2015) es el inmediatamente siguiente al último mes observado.

## 3. Validación de la estructura de los datos

### 3.a Tipos de datos de cada variable

In [9]:
tipo_semantico = {
    "date": "fecha (guardada como texto)",
    "date_block_num": "numérica ordinal (mes)",
    "shop_id": "categórica (identificador)",
    "item_id": "categórica (identificador)",
    "item_price": "numérica continua",
    "item_cnt_day": "numérica discreta",
    "item_name": "categórica (texto)",
    "item_category_id": "categórica (identificador)",
    "item_category_name": "categórica (texto)",
    "shop_name": "categórica (texto)",
    "ID": "categórica (identificador)",
}

pd.DataFrame(
    [
        {
            "tabla": nombre,
            "columna": columna,
            "dtype en pandas": str(df[columna].dtype),
            "tipo semántico": tipo_semantico[columna],
            "valores distintos": df[columna].nunique(),
        }
        for nombre, df in tablas.items()
        for columna in df.columns
    ]
).set_index(["tabla", "columna"])

dtype en pandas               tipo semántico  valores distintos
tabla           columna                                                                           
sales_train     date                           str  fecha (guardada como texto)               1034
                date_block_num               int64       numérica ordinal (mes)                 34
                shop_id                      int64   categórica (identificador)                 60
                item_id                      int64   categórica (identificador)              21807
                item_price                 float64            numérica continua              19993
                item_cnt_day               float64            numérica discreta                198
items           item_name                      str           categórica (texto)              22170
                item_id                      int64   categórica (identificador)              22170
                item_category_id             int64   categórica (identificador)                 84
item_categories item_category_name             str           categórica (texto)                 84
                item_category_id             int64   categórica (identificador)                 84
shops           shop_name                      str           categórica (texto)                 60
                shop_id                      int64   categórica (identificador)                 60
test            ID                           int64   categórica (identificador)             214200
                shop_id                      int64   categórica (identificador)                 42
                item_id                      int64   categórica (identificador)               5100

**Interpretación:** `date` se cargó como texto y debe convertirse a fecha. Los identificadores (`shop_id`, `item_id`, `item_category_id`) son enteros en pandas pero categóricos en significado, así que no tiene sentido promediarlos. En `sales_train` aparecen 21,807 de los 22,170 productos del catálogo, por lo que 363 nunca se vendieron en el histórico. Todas las 60 tiendas tienen ventas, mientras que `test` solo incluye 42 tiendas y 5,100 productos.

In [10]:
# item_cnt_day se carga como decimal: comprobamos si realmente tiene valores fraccionarios.
fraccionarios = (sales["item_cnt_day"] % 1 != 0).sum()
print(f"Valores no enteros en item_cnt_day: {fraccionarios:,}")

Valores no enteros en item_cnt_day: 0


**Interpretación:** `item_cnt_day` está guardada como decimal, pero no tiene ningún valor fraccionario. Es una cantidad entera y puede convertirse a entero sin perder información.

### 3.b Conversión de `date` a datetime

In [11]:
formato_valido = fecha_texto.str.fullmatch(r"\d{2}\.\d{2}\.\d{4}")

sales["date"] = pd.to_datetime(fecha_texto, format="%d.%m.%Y", errors="raise")

pd.Series(
    {
        "filas con formato dd.mm.aaaa": f"{formato_valido.sum():,} de {len(fecha_texto):,}",
        "dtype después de convertir": str(sales["date"].dtype),
        "fechas nulas (NaT)": sales["date"].isna().sum(),
        "fecha mínima": sales["date"].min().date(),
        "fecha máxima": sales["date"].max().date(),
        "archivo ordenado por fecha": sales["date"].is_monotonic_increasing,
    },
    name="resultado",
).to_frame()

,resultado
filas con formato dd.mm.aaaa,"2,935,849 de 2,935,849"
dtype después de convertir,datetime64[us]
fechas nulas (NaT),0
fecha mínima,2013-01-01
fecha máxima,2015-10-31
archivo ordenado por fecha,False


**Interpretación:** las 2,935,849 filas cumplen el formato `dd.mm.aaaa` y la conversión a datetime no generó fechas nulas, con el mismo rango hallado en 2.d. El archivo no está ordenado por fecha, de modo que hay que ordenar o agregar antes de cualquier operación que dependa de la secuencia temporal.

### 3.c Frecuencia temporal: días sin ventas, consistencia de `date_block_num` y frecuencia de agregación

**Días sin ventas a nivel de toda la cadena**

In [12]:
calendario = pd.date_range(sales["date"].min(), sales["date"].max(), freq="D")

diaria = (
    sales.groupby("date")
    .agg(
        unidades_netas=("item_cnt_day", "sum"),
        transacciones=("item_cnt_day", "size"),
        tiendas_activas=("shop_id", "nunique"),
    )
    .reindex(calendario, fill_value=0)
)

pd.Series(
    {
        "días en el calendario": len(calendario),
        "días con al menos un registro": (diaria["transacciones"] > 0).sum(),
        "días sin registros": (diaria["transacciones"] == 0).sum(),
        "días con unidades netas <= 0": (diaria["unidades_netas"] <= 0).sum(),
        "mínimo de tiendas activas en un día": diaria["tiendas_activas"].min(),
        "máximo de tiendas activas en un día": diaria["tiendas_activas"].max(),
    },
    name="valor",
).to_frame()

,valor
días en el calendario,1034
días con al menos un registro,1034
días sin registros,0
días con unidades netas <= 0,0
mínimo de tiendas activas en un día,17
máximo de tiendas activas en un día,52


**Interpretación:** a nivel de cadena no falta ningún día: los 1,034 días del calendario tienen registros y ninguno tiene unidades netas iguales o menores a cero. La serie diaria total está completa y no requiere rellenar fechas. En cambio, el número de tiendas activas por día varía entre 17 y 52 y nunca llega a 60, de modo que la cobertura por tienda no es constante.

In [13]:
diaria.nsmallest(5, "unidades_netas")

,unidades_netas,transacciones,tiendas_activas
2015-05-12,1274.0,1123,44
2015-05-13,1294.0,1128,44
2015-05-14,1302.0,1174,44
2015-10-05,1340.0,1176,41
2015-10-19,1341.0,1201,42


**Interpretación:** incluso los días más bajos registran más de 1,200 unidades netas, así que no hay días anómalamente vacíos. Los cinco mínimos ocurren en 2015, entre mayo y octubre, lo que coincide con la menor actividad de ese año que se observa en la tabla por mes más adelante.

**Días sin ventas por tienda**

In [14]:
inicio, fin = sales["date"].min(), sales["date"].max()

por_tienda = sales.groupby("shop_id").agg(
    primera_venta=("date", "min"),
    ultima_venta=("date", "max"),
    dias_con_venta=("date", "nunique"),
)
por_tienda["dias_en_su_periodo"] = (por_tienda["ultima_venta"] - por_tienda["primera_venta"]).dt.days + 1
por_tienda["dias_sin_venta"] = por_tienda["dias_en_su_periodo"] - por_tienda["dias_con_venta"]
por_tienda["pct_sin_venta"] = (por_tienda["dias_sin_venta"] / por_tienda["dias_en_su_periodo"] * 100).round(1)

pd.Series(
    {
        "tiendas con ventas": len(por_tienda),
        "tiendas activas de principio a fin": ((por_tienda["primera_venta"] == inicio) & (por_tienda["ultima_venta"] == fin)).sum(),
        "tiendas que empiezan después del primer día": (por_tienda["primera_venta"] > inicio).sum(),
        "tiendas que dejan de vender antes del último día": (por_tienda["ultima_venta"] < fin).sum(),
        "tiendas sin ningún hueco en su periodo": (por_tienda["dias_sin_venta"] == 0).sum(),
    },
    name="valor",
).to_frame()

,valor
tiendas con ventas,60
tiendas activas de principio a fin,12
tiendas que empiezan después del primer día,43
tiendas que dejan de vender antes del último día,19
tiendas sin ningún hueco en su periodo,11


**Interpretación:** solo 12 de las 60 tiendas venden desde el primer hasta el último día del histórico. De las otras 48, 43 empiezan después del primer día y 19 terminan antes del último (14 cumplen ambas condiciones). En varios casos la diferencia es mínima, como las tiendas 4, 10 y 24, que empiezan el 2 de enero de 2013, o la tienda 12, cuya última venta es del 30 de octubre de 2015. Solo 11 tiendas no tienen ningún hueco dentro de su propio periodo. Las series por tienda no son homogéneas y un día sin registros puede significar que la tienda aún no operaba, no que vendió cero.

In [15]:
por_tienda.sort_values("dias_sin_venta", ascending=False).head(10)

,primera_venta,ultima_venta,dias_con_venta,dias_en_su_periodo,dias_sin_venta,pct_sin_venta
shop_id,,,,,,
9,2013-10-03,2015-10-14,14,742,728,98.1
20,2014-10-02,2015-10-04,9,368,359,97.6
12,2013-01-03,2015-10-30,826,1031,205,19.9
27,2013-01-01,2015-09-30,899,1003,104,10.4
33,2014-08-29,2015-04-24,201,239,38,15.9
10,2013-01-02,2015-10-31,1001,1033,32,3.1
4,2013-01-02,2015-10-31,1010,1033,23,2.2
24,2013-01-02,2015-10-31,1012,1033,21,2.0
14,2013-01-01,2015-10-31,1014,1034,20,1.9


**Interpretación:** las tiendas 9 (`Выездная Торговля`, venta itinerante) y 20 (`Москва "Распродажа"`, liquidación) venden de forma esporádica: solo 14 y 9 días, con más de 97% de días sin venta dentro de su periodo. Son canales eventuales y conviene tratarlos por separado. La tienda 12 (`Интернет-магазин ЧС`, tienda en línea) tiene 19.9% de días sin venta, y el resto de esta lista está cerca o por debajo de 16%.

**Consistencia de `date_block_num` con la fecha**

In [16]:
bloque_esperado = (sales["date"].dt.year - 2013) * 12 + sales["date"].dt.month - 1
print(f"Filas donde date_block_num no coincide con la fecha: {(sales['date_block_num'] != bloque_esperado).sum():,}")

periodo = sales["date"].dt.to_period("M")
por_bloque = sales.groupby("date_block_num").agg(
    primera_fecha=("date", "min"),
    ultima_fecha=("date", "max"),
    dias_con_registros=("date", "nunique"),
    transacciones=("date", "size"),
)
por_bloque["meses_distintos"] = periodo.groupby(sales["date_block_num"]).nunique()
por_bloque["dias_del_mes"] = por_bloque["primera_fecha"].dt.days_in_month
por_bloque

Filas donde date_block_num no coincide con la fecha: 0


,primera_fecha,ultima_fecha,dias_con_registros,transacciones,meses_distintos,dias_del_mes
date_block_num,,,,,,
0,2013-01-01,2013-01-31,31,115690,1,31
1,2013-02-01,2013-02-28,28,108613,1,28
2,2013-03-01,2013-03-31,31,121347,1,31
3,2013-04-01,2013-04-30,30,94109,1,30
4,2013-05-01,2013-05-31,31,91759,1,31
5,2013-06-01,2013-06-30,30,100403,1,30
6,2013-07-01,2013-07-31,31,100548,1,31
7,2013-08-01,2013-08-31,31,104772,1,31
8,2013-09-01,2013-09-30,30,96137,1,30


**Interpretación:** `date_block_num` es totalmente consistente con la fecha: ninguna fila difiere del valor calculado, cada bloque corresponde a un único mes y cada mes tiene registros en todos sus días. Se puede usar con seguridad como índice mensual. Las transacciones suben con fuerza en diciembre (143,246 en 2013 y 130,786 en 2014) y caen entre años: octubre pasa de 94,202 en 2013 a 79,361 en 2014 y 53,514 en 2015.

**Comparación de frecuencias de agregación**

In [17]:
semana = sales["date"].dt.to_period("W")
dias_por_semana = sales.groupby(semana)["date"].nunique()
semanas_completas = dias_por_semana[dias_por_semana == 7].index

n_pares = sales[["shop_id", "item_id"]].drop_duplicates().shape[0]
niveles = {"diaria": sales["date"], "semanal": semana, "mensual": sales["date_block_num"]}

filas = []
for nombre, clave in niveles.items():
    serie = sales.groupby(clave)["item_cnt_day"].sum()
    if nombre == "semanal":
        serie = serie[serie.index.isin(semanas_completas)]
    observadas = pd.DataFrame(
        {"periodo": clave, "shop_id": sales["shop_id"], "item_id": sales["item_id"]}
    ).drop_duplicates().shape[0]
    posibles = clave.nunique() * n_pares
    filas.append(
        {
            "frecuencia": nombre,
            "periodos": clave.nunique(),
            "combinaciones periodo-tienda-producto posibles": posibles,
            "combinaciones con registros": observadas,
            "% con registros": round(observadas / posibles * 100, 2),
            "CV de las ventas totales": round(serie.std() / serie.mean(), 3),
        }
    )

print(f"Pares tienda-producto con historial: {n_pares:,}")
cruzan_mes = (semana.dt.start_time.dt.to_period("M") != semana.dt.end_time.dt.to_period("M"))
print(f"Semanas que cruzan dos meses: {semana[cruzan_mes].nunique()} de {semana.nunique()}")
pd.DataFrame(filas).set_index("frecuencia")

Pares tienda-producto con historial: 424,124


Semanas que cruzan dos meses: 30 de 148


,periodos,combinaciones periodo-tienda-producto posibles,combinaciones con registros,% con registros,CV de las ventas totales
frecuencia,,,,,
diaria,1034,438544216,2935821,0.67,0.449
semanal,148,62770352,2317130,3.69,0.317
mensual,34,14420216,1609124,11.16,0.265


**Interpretación y conclusión:** conviene agregar a frecuencia **mensual**, por cuatro razones basadas en estos resultados:

1. La variable objetivo y el periodo a predecir son mensuales, y `date_block_num` ofrece un índice mensual sin errores.
2. Por combinación periodo-tienda-producto, solo 0.67% tiene registros a nivel diario y 3.69% a nivel semanal, frente a 11.16% a nivel mensual.
3. La variabilidad relativa de las ventas totales baja al agregar: el coeficiente de variación pasa de 0.449 (diaria) a 0.317 (semanal) y 0.265 (mensual).
4. La semana no encaja con el objetivo: 30 de las 148 semanas cruzan dos meses.

La limitación es que solo hay 34 observaciones mensuales, menos de tres ciclos anuales completos. Por eso la serie total también debe analizarse a nivel diario, como recomienda el PDF para la sección 9, y para estudiar los ciclos por día de la semana. Aun a nivel mensual, cerca de 89% de las combinaciones no tiene registros, lo que habrá que tener en cuenta al preparar el modelo.

## 4. Análisis de valores faltantes y duplicados

### 4.a Valores nulos y porcentaje de faltantes

In [18]:
pd.DataFrame(
    [
        {
            "tabla": nombre,
            "columna": columna,
            "filas": len(df),
            "nulos": df[columna].isna().sum(),
            "% faltantes": round(df[columna].isna().mean() * 100, 4),
        }
        for nombre, df in tablas.items()
        for columna in df.columns
    ]
).set_index(["tabla", "columna"])

filas  nulos  % faltantes
tabla           columna                                        
sales_train     date                2935849      0          0.0
                date_block_num      2935849      0          0.0
                shop_id             2935849      0          0.0
                item_id             2935849      0          0.0
                item_price          2935849      0          0.0
                item_cnt_day        2935849      0          0.0
items           item_name             22170      0          0.0
                item_id               22170      0          0.0
                item_category_id      22170      0          0.0
item_categories item_category_name       84      0          0.0
                item_category_id         84      0          0.0
shops           shop_name                60      0          0.0
                shop_id                  60      0          0.0
test            ID                   214200      0          0.0
                shop_id              214200      0          0.0
                item_id              214200      0          0.0

**Interpretación:** ninguna columna de las cinco tablas tiene valores nulos (0% de faltantes en todas). No hace falta eliminar ni imputar por este motivo. Los problemas de calidad del dataset no son celdas vacías, sino valores inválidos y registros repetidos, que se revisan a continuación.

### 4.b Valores inválidos: precios <= 0 y cantidades negativas

**Precios menores o iguales a cero**

In [19]:
precios_invalidos = sales[sales["item_price"] <= 0]
print(f"Filas con item_price <= 0: {len(precios_invalidos):,}")
print(f"Filas con item_price = 0: {(sales['item_price'] == 0).sum():,}")
precios_invalidos

Filas con item_price <= 0: 1
Filas con item_price = 0: 0


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
484683,2013-05-15,4,32,2973,-1.0,1.0


**Interpretación:** solo una fila de 2,935,849 tiene precio inválido: `item_price = -1` para el producto 2973 en la tienda 32, el 15 de mayo de 2013. No hay precios iguales a cero. La cantidad de esa fila es +1, así que es una venta normal con un precio mal registrado y conviene corregir el precio en lugar de eliminar la venta.

In [20]:
# Referencia: precios válidos del mismo producto, en la misma tienda y en toda la cadena.
filas_contexto = []
for _, fila in precios_invalidos.iterrows():
    mismo_item = sales[(sales["item_id"] == fila["item_id"]) & (sales["item_price"] > 0)]
    misma_tienda = mismo_item[mismo_item["shop_id"] == fila["shop_id"]]
    for alcance, datos in {"producto y tienda": misma_tienda, "producto (toda la cadena)": mismo_item}.items():
        filas_contexto.append(
            {
                "item_id": fila["item_id"],
                "shop_id": fila["shop_id"],
                "alcance": alcance,
                "transacciones": len(datos),
                "precio mínimo": datos["item_price"].min(),
                "precio mediano": datos["item_price"].median(),
                "precio máximo": datos["item_price"].max(),
            }
        )
pd.DataFrame(filas_contexto)

,item_id,shop_id,alcance,transacciones,precio mínimo,precio mediano,precio máximo
0,2973,32,producto y tienda,12,1249.0,2499.0,2499.0
1,2973,32,producto (toda la cadena),779,1049.0,2499.0,2499.0


**Interpretación:** el producto 2973 tiene 12 transacciones con precio válido en la tienda 32 y 779 en toda la cadena. En ambos casos la mediana es 2,499, que además coincide con el precio máximo. Hay suficiente historial en la misma tienda para usar la mediana producto-tienda.

In [21]:
invalido = sales["item_price"] <= 0
validos = sales.loc[~invalido]

mediana_item_tienda = validos.groupby(["item_id", "shop_id"])["item_price"].median().rename("mediana_item_tienda")
mediana_item = validos.groupby("item_id")["item_price"].median().rename("mediana_item")

correccion = (
    sales.loc[invalido, ["date", "shop_id", "item_id", "item_price"]]
    .join(mediana_item_tienda, on=["item_id", "shop_id"])
    .join(mediana_item, on="item_id")
)
correccion["precio_corregido"] = correccion["mediana_item_tienda"].fillna(correccion["mediana_item"])
correccion["criterio"] = (
    correccion["mediana_item_tienda"].notna().map({True: "mediana producto-tienda", False: "mediana producto"})
)

sales.loc[correccion.index, "item_price"] = correccion["precio_corregido"]

print(f"Filas con item_price <= 0 después de corregir: {(sales['item_price'] <= 0).sum():,}")
correccion

Filas con item_price <= 0 después de corregir: 0


,date,shop_id,item_id,item_price,mediana_item_tienda,mediana_item,precio_corregido,criterio
484683,2013-05-15,32,2973,-1.0,2499.0,2499.0,2499.0,mediana producto-tienda


**Interpretación:** el precio inválido se reemplazó por 2,499, la mediana del mismo producto en la misma tienda, y ya no quedan precios menores o iguales a cero. Se usa la mediana porque no se ve afectada por los descuentos puntuales, como el precio mínimo de 1,249 en esa tienda. La mediana solo por producto queda como respaldo, pero aquí no fue necesaria.

**Cantidades negativas (devoluciones)**

In [22]:
devoluciones = sales[sales["item_cnt_day"] < 0]
unidades_devueltas = -devoluciones["item_cnt_day"].sum()
unidades_brutas = sales.loc[sales["item_cnt_day"] > 0, "item_cnt_day"].sum()

primera_venta = (
    sales[sales["item_cnt_day"] > 0].groupby(["shop_id", "item_id"])["date"].min().rename("primera_venta")
)
con_venta_previa = devoluciones.join(primera_venta, on=["shop_id", "item_id"])
con_venta_previa = (con_venta_previa["date"] >= con_venta_previa["primera_venta"]).sum()

display(
    pd.Series(
        {
            "filas con item_cnt_day < 0": f"{len(devoluciones):,}",
            "% de las filas": f"{len(devoluciones) / len(sales) * 100:.2f}%",
            "filas con item_cnt_day = 0": f"{(sales['item_cnt_day'] == 0).sum():,}",
            "unidades devueltas": f"{unidades_devueltas:,.0f}",
            "unidades vendidas brutas": f"{unidades_brutas:,.0f}",
            "% devuelto sobre lo vendido": f"{unidades_devueltas / unidades_brutas * 100:.2f}%",
            "devoluciones con venta previa del mismo par": f"{con_venta_previa:,} ({con_venta_previa / len(devoluciones) * 100:.1f}%)",
        },
        name="valor",
    ).to_frame()
)
devoluciones["item_cnt_day"].value_counts().sort_index(ascending=False).head(10).to_frame("filas")

,valor
filas con item_cnt_day < 0,"7,356"
% de las filas,0.25%
filas con item_cnt_day = 0,0
unidades devueltas,"7,541"
unidades vendidas brutas,"3,655,747"
% devuelto sobre lo vendido,0.21%
devoluciones con venta previa del mismo par,"7,230 (98.3%)"


,filas
item_cnt_day,
-1.0,7252
-2.0,78
-3.0,14
-4.0,3
-5.0,4
-6.0,2
-9.0,1
-16.0,1
-22.0,1


**Interpretación:** hay 7,356 filas con cantidad negativa (0.25%), equivalentes a 7,541 unidades, es decir, 0.21% de lo vendido. Casi todas son de una sola unidad (7,252 filas con -1), aunque existen casos de hasta -22. El 98.3% ocurre en un par tienda-producto que ya tenía una venta en esa fecha o antes, lo que respalda que son devoluciones reales. No existen filas con cantidad cero.

In [23]:
# Efecto sobre la variable objetivo si se eliminaran las devoluciones.
clave_mensual = ["date_block_num", "shop_id", "item_id"]
neta = sales.groupby(clave_mensual)["item_cnt_day"].sum()
sin_devoluciones = (
    sales[sales["item_cnt_day"] >= 0].groupby(clave_mensual)["item_cnt_day"].sum().reindex(neta.index, fill_value=0)
)

pd.Series(
    {
        "combinaciones mes-tienda-producto": f"{len(neta):,}",
        "cuyo total cambiaría sin devoluciones": f"{(neta != sin_devoluciones).sum():,}",
        "con total mensual neto negativo": f"{(neta < 0).sum():,}",
        "con total mensual neto igual a cero": f"{(neta == 0).sum():,}",
        "unidades netas (con devoluciones)": f"{neta.sum():,.0f}",
        "unidades sin devoluciones": f"{sin_devoluciones.sum():,.0f}",
    },
    name="valor",
).to_frame()

,valor
combinaciones mes-tienda-producto,"1,609,124"
cuyo total cambiaría sin devoluciones,"7,259"
con total mensual neto negativo,915
con total mensual neto igual a cero,"2,583"
unidades netas (con devoluciones),"3,648,206"
unidades sin devoluciones,"3,655,747"


**Interpretación y justificación:** las devoluciones se conservan como ventas negativas, sin modificarlas. La variable objetivo es la suma mensual de `item_cnt_day`, de modo que una devolución debe restar de las unidades efectivamente vendidas. Si se eliminaran, 7,259 combinaciones mes-tienda-producto cambiarían de valor y el total subiría de 3,648,206 a 3,655,747 unidades, sobrestimando la demanda neta. Además, 915 combinaciones quedan con total mensual negativo y 2,583 en cero, casos que la etapa de modelado deberá considerar.

### 4.c Registros duplicados y tiendas repetidas

**Filas duplicadas en `sales_train`**

In [24]:
duplicadas = sales[sales.duplicated(keep=False)].sort_values(list(sales.columns))
n_sobrantes = sales.duplicated().sum()
print(f"Filas involucradas en duplicados exactos: {len(duplicadas):,}")
print(f"Filas sobrantes a eliminar: {n_sobrantes:,}")
duplicadas

Filas involucradas en duplicados exactos: 12
Filas sobrantes a eliminar: 6


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
76961,2013-01-05,0,54,20130,149.0,1.0
76962,2013-01-05,0,54,20130,149.0,1.0
1435365,2014-02-23,13,50,3423,999.0,1.0
1435367,2014-02-23,13,50,3423,999.0,1.0
1496765,2014-03-23,14,21,3423,999.0,1.0
1496766,2014-03-23,14,21,3423,999.0,1.0
1671872,2014-05-01,16,50,3423,999.0,1.0
1671873,2014-05-01,16,50,3423,999.0,1.0
1866322,2014-07-12,18,25,3423,999.0,1.0
1866340,2014-07-12,18,25,3423,999.0,1.0


**Interpretación:** existen 6 parejas de filas exactamente iguales (12 filas en total), todas con una unidad vendida; cuatro corresponden al producto 3423. El PDF define el archivo como una fila por producto vendido en una tienda y un día, y estas filas repiten la fecha, la tienda, el producto, el precio y la cantidad. Por eso se consideran registros repetidos y se conserva una fila de cada pareja, eliminando 6. El efecto es mínimo, pero evita contar dos veces la misma venta.

In [25]:
sales = sales.drop_duplicates(keep="first")
tablas["sales_train"] = sales

clave_diaria = ["date", "shop_id", "item_id"]
misma_clave = sales[sales.duplicated(clave_diaria, keep=False)].sort_values(clave_diaria)

print(f"Duplicados exactos restantes: {sales.duplicated().sum():,}")
print(f"Filas que comparten fecha, tienda y producto, pero difieren en precio o cantidad: {len(misma_clave):,}")
misma_clave.head(10)

Duplicados exactos restantes: 0
Filas que comparten fecha, tienda y producto, pero difieren en precio o cantidad: 44


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
70686,2013-01-25,0,31,14050,349.0,1.0
70718,2013-01-25,0,31,14050,248.0,1.0
103461,2013-01-27,0,38,15702,549.0,1.0
103463,2013-01-27,0,38,15702,149.0,-1.0
6959,2013-01-29,0,25,12133,889.0,1.0
6960,2013-01-29,0,25,12133,1389.0,1.0
210156,2013-02-16,1,25,12133,1389.0,1.0
210173,2013-02-16,1,25,12133,889.0,1.0
142696,2013-02-18,1,16,15702,149.0,1.0
142697,2013-02-18,1,16,15702,549.0,1.0


**Interpretación:** ya no quedan duplicados exactos. Otras 44 filas comparten fecha, tienda y producto, pero difieren en el precio o en la cantidad, por ejemplo una venta a 549 y una devolución a 149 el mismo día. Estas filas se conservan porque contienen información distinta y no hay forma de saber que alguna sea errónea. Pueden ser ventas a precios diferentes o movimientos separados del mismo día.

In [26]:
pd.DataFrame(
    [{"tabla": nombre, "filas": len(df), "filas duplicadas": df.duplicated().sum()} for nombre, df in tablas.items()]
).set_index("tabla")

,filas,filas duplicadas
tabla,,
sales_train,2935843,0
items,22170,0
item_categories,84,0
shops,60,0
test,214200,0


**Interpretación:** los catálogos y `test` no tienen filas duplicadas, y `sales_train` queda con 2,935,843 filas. La repetición de tiendas, por tanto, no se ve al comparar filas completas: hay que buscarla comparando los nombres.

**Tiendas repetidas en `shops.csv`**

In [27]:
import re
from difflib import SequenceMatcher
from itertools import combinations


def normalizar(nombre):
    """Quita mayúsculas, signos y la marca 'фран' para comparar nombres de tiendas."""
    texto = re.sub(r"фран|[^\w\s]", " ", nombre.lower())
    return " ".join(texto.split())


nombres = shops.set_index("shop_id")["shop_name"]
similares = []
for a, b in combinations(nombres.index, 2):
    similitud = SequenceMatcher(None, normalizar(nombres[a]), normalizar(nombres[b])).ratio()
    if similitud >= 0.8:
        similares.append(
            {"shop_id A": a, "shop_id B": b, "nombre A": nombres[a], "nombre B": nombres[b], "similitud": round(similitud, 3)}
        )

print(f"Nombres exactamente repetidos: {shops['shop_name'].duplicated().sum()}")
pd.DataFrame(similares).sort_values("similitud", ascending=False).reset_index(drop=True)

Nombres exactamente repetidos: 0


,shop_id A,shop_id B,nombre A,nombre B,similitud
0,0,57,"!Якутск Орджоникидзе, 56 фран","Якутск Орджоникидзе, 56",1.000
1,1,58,"!Якутск ТЦ ""Центральный"" фран","Якутск ТЦ ""Центральный""",1.000
2,10,11,Жуковский ул. Чкалова 39м?,Жуковский ул. Чкалова 39м²,0.980
3,23,24,"Москва ТК ""Буденовский"" (пав.А2)","Москва ТК ""Буденовский"" (пав.К7)",0.929
4,39,40,"РостовНаДону ТРК ""Мегацентр Горизонт""","РостовНаДону ТРК ""Мегацентр Горизонт"" Островной",0.875
5,30,31,"Москва ТЦ ""Перловский""","Москва ТЦ ""Семеновский""",0.829
6,38,54,"Омск ТЦ ""Мега""","Химки ТЦ ""Мега""",0.800


**Interpretación:** ningún nombre se repite de forma idéntica, pero al normalizar el texto aparecen las parejas buscadas. Las parejas 0-57 y 1-58 tienen el mismo nombre una vez quitados el signo `!` y la marca `фран`, y 10-11 solo difiere en un carácter final (`?` frente a `²`). La pareja 39-40 es parecida, aunque 40 agrega `Островной`. Las demás coincidencias no apuntan a la misma tienda: 23-24 son pabellones distintos (`А2` y `К7`), mientras que 30-31 y 38-54 tienen distinto centro comercial o ciudad.

In [28]:
parejas = [(0, 57), (1, 58), (10, 11), (39, 40)]
meses_tienda = sales.groupby("shop_id")["date_block_num"].agg(lambda s: set(s.astype(int)))
tiendas_test = set(test["shop_id"])


def rango(meses):
    return f"{min(meses)}-{max(meses)} ({len(meses)} meses)"


filas_parejas = []
for a, b in parejas:
    traslape = sorted(meses_tienda[a] & meses_tienda[b])
    filas_parejas.append(
        {
            "shop_id A": a,
            "shop_id B": b,
            "meses A": rango(meses_tienda[a]),
            "meses B": rango(meses_tienda[b]),
            "meses traslapados": len(traslape),
            "lista de traslape": traslape,
            "A en test": a in tiendas_test,
            "B en test": b in tiendas_test,
        }
    )
actividad_parejas = pd.DataFrame(filas_parejas)
actividad_parejas

,shop_id A,shop_id B,meses A,meses B,meses traslapados,lista de traslape,A en test,B en test
0,0,57,0-1 (2 meses),2-33 (32 meses),0,[],False,True
1,1,58,0-1 (2 meses),2-33 (32 meses),0,[],False,True
2,10,11,0-33 (33 meses),25-25 (1 meses),0,[],True,False
3,39,40,14-33 (20 meses),14-24 (11 meses),11,"[14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]",True,False


**Interpretación:** las parejas 0-57, 1-58 y 10-11 no tienen ningún mes en común, y en cada una exactamente un ID aparece en `test`, que será el ID de destino. En cambio, 39 y 40 **sí se traslapan en 11 meses** (bloques 14 a 24), así que no se unifican: el criterio de no traslape no se cumple y funcionaron como puntos de venta simultáneos.

In [29]:
ids_parejas = [tienda for pareja in parejas for tienda in pareja]
transacciones_mes = (
    sales[sales["shop_id"].isin(ids_parejas)]
    .pivot_table(index="shop_id", columns="date_block_num", values="item_cnt_day", aggfunc="size", fill_value=0)
    .reindex(ids_parejas)
)
transacciones_mes

date_block_num,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33
shop_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,4793,5064,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
57,0,0,5518,4890,4613,4876,5065,4615,4298,4357,4148,6701,4048,4160,4076,3564,3279,3325,3314,3331,3054,3243,2987,5133,3460,3005,2797,2464,2097,2234,2155,2535,1991,2095
1,2694,2984,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
58,0,0,3360,2613,2150,2592,2799,2613,2406,2551,2489,4434,2305,2491,2544,2020,1771,1933,2003,2427,1939,1825,2013,3868,2146,2064,2110,1492,1269,1457,1607,1612,1210,1328
10,875,912,888,699,630,681,746,681,798,829,808,1174,698,764,761,543,583,525,513,584,603,567,611,955,564,0,507,459,364,423,409,409,442,392
11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,499,0,0,0,0,0,0,0,0
39,0,0,0,0,0,0,0,0,0,0,0,0,0,0,745,584,709,624,560,648,564,543,735,1134,780,642,741,556,636,545,578,757,664,695
40,0,0,0,0,0,0,0,0,0,0,0,0,0,0,407,278,274,293,328,310,338,358,441,825,405,0,0,0,0,0,0,0,0,0


**Interpretación:** el relevo entre IDs se ve con claridad. Las tiendas 0 y 1 venden solo en los bloques 0 y 1, y justo desde el bloque 2 empiezan 57 y 58. La tienda 11 solo tiene ventas en el bloque 25, el único mes en que la tienda 10 tiene cero. Son los mismos negocios registrados con otro ID. Las tiendas 39 y 40, en cambio, venden a la vez en cada mes del bloque 14 al 24, y unirlas sumaría dos operaciones paralelas.

In [30]:
# Solo se unifica una pareja sin meses traslapados y con exactamente un ID presente en test.
mapa_tiendas = {}
for _, fila in actividad_parejas.iterrows():
    a, b = int(fila["shop_id A"]), int(fila["shop_id B"])
    if fila["meses traslapados"] == 0 and (a in tiendas_test) != (b in tiendas_test):
        destino, origen = (a, b) if a in tiendas_test else (b, a)
        mapa_tiendas[origen] = destino

print(f"Unificación aplicada (origen -> destino): {mapa_tiendas}")

filas_sales_reasignadas = sales["shop_id"].isin(mapa_tiendas).sum()
filas_test_reasignadas = test["shop_id"].isin(mapa_tiendas).sum()

sales["shop_id"] = sales["shop_id"].replace(mapa_tiendas)
test["shop_id"] = test["shop_id"].replace(mapa_tiendas)
shops = shops[~shops["shop_id"].isin(mapa_tiendas)]
tablas["shops"] = shops
tablas["test"] = test

pd.Series(
    {
        "filas de sales_train reasignadas": f"{filas_sales_reasignadas:,}",
        "filas de test reasignadas": f"{filas_test_reasignadas:,}",
        "tiendas en shops": shops["shop_id"].nunique(),
        "tiendas con ventas": sales["shop_id"].nunique(),
        "tiendas en test": test["shop_id"].nunique(),
        "tiendas de test sin ventas ni catálogo": len(set(test["shop_id"]) - set(sales["shop_id"]) - set(shops["shop_id"])),
        "duplicados exactos en sales_train": sales.duplicated().sum(),
        "pares duplicados en test": test.duplicated(["shop_id", "item_id"]).sum(),
    },
    name="valor",
).to_frame()

Unificación aplicada (origen -> destino): {0: 57, 1: 58, 11: 10}


,valor
filas de sales_train reasignadas,"16,034"
filas de test reasignadas,0
tiendas en shops,57
tiendas con ventas,57
tiendas en test,42
tiendas de test sin ventas ni catálogo,0
duplicados exactos en sales_train,0
pares duplicados en test,0


**Interpretación:** se aplicó 0→57, 1→58 y 11→10, lo que reasignó 16,034 filas de `sales_train`. En `test` se aplicó el mismo mapeo, pero cambió 0 filas porque ya usa únicamente los IDs de destino. El catálogo queda con 57 tiendas, todas las tiendas de `test` existen en el histórico y la unificación no generó duplicados. Las tres tiendas unificadas ahora tienen una serie continua, en lugar de un historial partido en dos.

### Resumen de la limpieza de la sección 4

In [31]:
# Se compara contra los archivos originales, que no se modificaron.
sales_original = pd.read_csv(RAW / "sales_train.csv")
test_original = pd.read_csv(RAW / "test.csv")
shops_original = pd.read_csv(RAW / "shops.csv")

conservadas = sales_original.loc[sales.index]
sobrantes_original = sales_original.duplicated()

display(
    pd.DataFrame(
        [
            ["sales_train", "item_price <= 0 corregido con mediana", "modificada", (conservadas["item_price"] != sales["item_price"]).sum()],
            ["sales_train", "shop_id unificado", "modificada", (conservadas["shop_id"] != sales["shop_id"]).sum()],
            ["sales_train", "duplicado exacto eliminado", "eliminada", len(sales_original) - len(sales)],
            ["sales_train", "devolución conservada (sin cambios)", "conservada", (sales["item_cnt_day"] < 0).sum()],
            ["test", "shop_id unificado", "modificada", (test_original["shop_id"] != test["shop_id"]).sum()],
            ["shops", "tienda repetida eliminada del catálogo", "eliminada", len(shops_original) - len(shops)],
        ],
        columns=["tabla", "acción", "tipo", "filas"],
    )
)

display(
    pd.DataFrame(
        [
            {"tabla": "sales_train", "filas antes": len(sales_original), "filas después": len(sales)},
            {"tabla": "test", "filas antes": len(test_original), "filas después": len(test)},
            {"tabla": "shops", "filas antes": len(shops_original), "filas después": len(shops)},
        ]
    ).set_index("tabla")
)

unidades_antes = sales_original["item_cnt_day"].sum()
unidades_despues = sales["item_cnt_day"].sum()
print(f"Unidades netas antes: {unidades_antes:,.0f} | después: {unidades_despues:,.0f}")
print(f"Diferencia: {unidades_antes - unidades_despues:,.0f} | unidades en las filas eliminadas: {sales_original.loc[sobrantes_original, 'item_cnt_day'].sum():,.0f}")

,tabla,acción,tipo,filas
0,sales_train,item_price <= 0 corregido con mediana,modificada,1
1,sales_train,shop_id unificado,modificada,16034
2,sales_train,duplicado exacto eliminado,eliminada,6
3,sales_train,devolución conservada (sin cambios),conservada,7356
4,test,shop_id unificado,modificada,0
5,shops,tienda repetida eliminada del catálogo,eliminada,3


,filas antes,filas después
tabla,,
sales_train,2935849,2935843
test,214200,214200
shops,60,57


Unidades netas antes: 3,648,206 | después: 3,648,200
Diferencia: 6 | unidades en las filas eliminadas: 6


**Interpretación:** la limpieza fue muy acotada. En `sales_train` se corrigió 1 precio, se reasignó el `shop_id` de 16,034 filas y se eliminaron 6 duplicados, pasando de 2,935,849 a 2,935,843 filas. Las 7,356 devoluciones se conservaron y `test` no sufrió cambios. La verificación contra los archivos originales cuadra: las unidades netas bajan exactamente en 6, las de las filas eliminadas. Estos cambios solo existen en memoria durante el notebook, y los archivos de `data/raw/` no se modificaron.